In [1]:
import polars as pl
import torch


players_df = pl.read_csv('csv/player.csv')

all_ids = players_df['id'].unique()
null_player_id = 0
nba_id_to_idx_mapping = {}
for idx, nba_id in enumerate(all_ids):
    nba_id_to_idx_mapping[nba_id] = idx + 1 # reserve 0 for null player id

nba_id_to_idx_mapping[null_player_id] = 0

max_nba_id = max(nba_id_to_idx_mapping.keys())
lookup_tensor = torch.zeros(max_nba_id + 1, dtype=torch.long)
for nba_id, sequential_idx in nba_id_to_idx_mapping.items():
    lookup_tensor[nba_id] = sequential_idx
    

In [11]:
from datasets import Dataset, load_dataset
from torch.utils.data import DataLoader

REPO_ID = 'sviridov/nba-posession-processed'
processed_dataset = load_dataset(REPO_ID, split='train')

processed_dataset.set_format("torch")

loader = DataLoader(
    processed_dataset, 
    batch_size=2048,      # Critical for GPU utilization
    shuffle=True, 
    num_workers=4,         
    pin_memory=True,       
    persistent_workers=True # Keeps data loading alive between epochs
)


print(len(loader) * 2048, " training shot cycles")

5349376  training shot cycles


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [ ]:
from NBADeepFm import NBATransformer

model = NBATransformer(num_players=len(nba_id_to_idx_mapping), embed_dim=64)
model = model.to(device)


In [ ]:
# train the model
from torch.amp import autocast, GradScaler
from torch import nn

scaler = GradScaler()
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

num_players = len(nba_id_to_idx_mapping)
MASK_ID = num_players # Use the last index as the mask
MASK_ID = torch.tensor(MASK_ID, dtype=torch.long).to(device)

num_epochs = 5
batch_size = 64

blank_hwp = torch.tensor([0.5, 0.5, 3], dtype=torch.float32, device=device)




for epoch in range(num_epochs):
    for i, batch in enumerate(loader):
        
        lineup_ids = batch['lineup_ids'].to(device)
        lineup_hwps = batch['lineup_hwps'].to(device)
        
        is_putback = batch['is_putback'].to(device)
        is_freethrow = batch['is_freethrow'].to(device)
        shot_distance = batch['shot_distance'].to(device)

        role_ids = torch.stack([
            batch['shooting_player'], 
            batch['assisting_player'], 
            batch['defending_player'], 
            batch['rebounding_player']
        ], dim=1).to(device)
        
        role_hwps = torch.stack([
            batch['shooting_hwp'], 
            batch['assisting_hwp'], 
            batch['defending_hwp'], 
            batch['rebounding_hwp']
        ], dim=1).to(device)
        
        # randomly mask shooter 50% of time; assister, rebounder, and defender 17% of the time
        #only mask one per training sample
        # --- THE MASKING LOGIC ---
        curr_batch_size = role_ids.size(0)
        
        # Generate random numbers between 0.0 and 1.0 for the whole batch
        rand_probs = torch.rand(curr_batch_size, device=device)
        
        # Create a tensor to hold our chosen indices (default is 0: Shooter)
        mask_indices = torch.zeros(curr_batch_size, dtype=torch.long, device=device)
        
        # Overwrite indices based on your probabilities
        mask_indices[(rand_probs >= 0.50) & (rand_probs < 0.66)] = 1  # Assister
        mask_indices[(rand_probs >= 0.66) & (rand_probs < 0.83)] = 2  # Defender
        mask_indices[rand_probs >= 0.83] = 3                          # Rebounder
        
        # Create an array of batch indices [0, 1, 2, ... batch_size-1]
        b_idx = torch.arange(curr_batch_size, device=device)
        
        # Save the real player IDs as the target labels BEFORE we mask them
        true_labels = role_ids[b_idx, mask_indices].clone()
        
        # Apply the MASK_ID to the chosen roles
        role_ids[b_idx, mask_indices] = MASK_ID
        
        # Apply the blank physical profile to the chosen roles to prevent leakage
        role_hwps[b_idx, mask_indices] = blank_hwp
        

        # mask the player in the lineup too
        mask_matches = (lineup_ids == true_labels.unsqueeze(1))
        lineup_ids[mask_matches] = MASK_ID
        lineup_hwps[mask_matches] = blank_hwp


        # ----------------------------

        optimizer.zero_grad()

        with autocast(device_type=device):
            id_logits = model(
                lineup_ids, lineup_hwps, 
                role_ids, role_hwps, 
                is_putback, is_freethrow, shot_distance,
                mask_indices
            )
            loss = criterion(id_logits, true_labels)            



        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        if (i+1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(loader)}], Loss: {loss.item():.4f}')
    
    
    torch.save(model.state_dict(), f"nba_transformer_epoch{epoch}_checkpoint.pth") # Save state_dict, not whole model
            


print("Finished training model")

/tmp/ipykernel_140693/664922028.py:5: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler()


NameError: name 'device' is not defined

In [ ]:
#upload to hugging face hub


# model.push_to_hub("sviridov/nba-transformer", private=True)